# Instructional Notebook for SHRED + Biomechanics

The following lines can be uncommented if running this notebook in Google Colab. Uncomment by highlighting lines and pressing Ctrl+/

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')
#!pip install mat73
#!git clone https://github.com/Jan-Williams/pyshred
#%cd /content/pyshred

These lines import standard packages for managing data.

In [2]:
import os
import numpy as np
import altair as alt
import pandas as pd
from processdata import TimeSeriesDataset
import models_TCN
import torch
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
import mat73
import functions as ft

# *** Update ***
Load a subject's data and manipulate the dataframe to be "tidy" = one row per time step and one column per signal. Depending on the dataset, loading and managing data will look different.

## Obtain subject's experimental data

Items to adjust before running a trial:

*   save_df - True (save) or False (don't save)
*   subject - '##'
*   activity code - AC## (this may need a different identifier depending on the dataset, just need a way to distinguish running speeds)

In [3]:
# Change subject number
subj = '01'    # 01-09
save_df = True   # True: save SHRED output dataframes only, False: don't save
trial_length = 5  # in minutes, accepts integers 1-6
frequency = 128 # in Hz, accepts integers up to 128

# adjust file path for saving if parameters are modified from 6min or 128Hz
if trial_length == 6:
    save_tag = str(frequency)+'Hz'
elif frequency == 128:
    save_tag = str(trial_length)+'min'


## Import Matlab file structure with subject's experimental data

Access directories where data is stored and will be saved. Manually set up folders before running the code block to ensure known file paths.

In [4]:
cwd = os.getcwd()
main_path = os.path.dirname(cwd) + '/Datasets' 

# Alternatively, use the below lines if using Colab
# main_path = '/content/drive/MyDrive/Colab_Notebooks/Datasets'
# main_path = cwd+'/Datasets'

dataset_path = main_path+'/Data' # sets path to dataset / raw data
dataframe_path = main_path+'/Dataframes'  # file path for saved dataframe results of test data
figure_path = main_path+'/Figures'
model_path = main_path+'/Models' # optionally, save the models that are trained
print(dataset_path)
print(dataframe_path)


/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Data
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes


Note on changing file directory and needing to save the parent directory:

https://stackoverflow.com/questions/14462833/how-can-i-go-back-to-the-previous-working-directory-after-changing-it

In [5]:
# load .mat file into pandas dataframe
load_mat = mat73.loadmat(dataset_path+'/Subject'+subj+'.mat')['Subject'+subj]
df = pd.DataFrame.from_dict(load_mat)

The example dataset contains several activities, two of which are 'Walking' and 'Running'. Access each independently.

In [6]:
#df_tmp = pd.DataFrame(data=df['Walking']['APDM_Accel']['Data'],
#                      columns = df['Walking']['APDM_Accel']['Labels'])

df_tmp = pd.DataFrame(data=df['Running']['APDM_Accel']['Data'],
                      columns = df['Running']['APDM_Accel']['Labels'])

pd.set_option('display.max_columns', None)

df_tmp.head(5) # check that correct data was selected

Time (s) Activity Code                Waist                      \
                          Acceleration (m/s^2)                       
                                             x         y         z   
0  0.007812          22.0            -9.846344  0.171822  1.227401   
1  0.015624          22.0            -9.775612  0.050246  1.209962   
2  0.023437          22.0            -9.835819  0.125845  1.167702   
3  0.031249          22.0            -9.810564  0.107856  1.203309   
4  0.039061          22.0            -9.800482  0.050737  1.143245   

                                                                               \
  Angular Velocity (rad/s)                     Magnetic Field (uT)              
                         x         y         z                   x          y   
0                 0.091794 -0.027342 -0.014275           46.815190  16.071594   
1                 0.094758 -0.032213 -0.006748           46.996207  16.216266   
2                 0.097836 -0.038619 -0.010032           46.728334  16.212284   
3                 0.086923 -0.035422 -0.006915           46.646590  16.073072   
4                 0.091569 -0.041706 -0.007019           46.688921  16.195991   

                            Chest                      \
             Acceleration (m/s^2)                       
           z                    x         y         z   
0  13.902695            -9.764181  1.311812  0.985503   
1  13.923646            -9.757018  1.316749  0.992521   
2  13.792188            -9.754793  1.332469  0.997156   
3  13.775756            -9.757444  1.331871  0.987872   
4  13.996201            -9.746306  1.337988  0.989957   

                                                                              \
  Angular Velocity (rad/s)                     Magnetic Field (uT)             
                         x         y         z                   x         y   
0                 0.070709  0.084543  0.012162           55.879245 -1.209833   
1                 0.067192  0.088808  0.013694           55.828066 -1.158919   
2                 0.065436  0.091731  0.010487           55.859044 -1.419488   
3                 0.081388  0.086023  0.012167           55.757635 -1.280665   
4                 0.081584  0.084634  0.013785           55.632360 -1.164697   

                       Left Ankle                      \
             Acceleration (m/s^2)                       
           z                    x         y         z   
0  11.792871            -9.741885 -0.468814 -0.882530   
1  11.847663            -9.729948 -0.520553 -0.887037   
2  11.842671            -9.728399 -0.466783 -0.891656   
3  11.819464            -9.723669 -0.453419 -0.884954   
4  11.939039            -9.716266 -0.484920 -0.887274   

                                                                               \
  Angular Velocity (rad/s)                     Magnetic Field (uT)              
                         x         y         z                   x          y   
0                 0.118064  0.017438  0.021202           31.514212  18.934442   
1                 0.113823  0.015991  0.017926           31.112110  18.884349   
2                 0.116528  0.015885  0.019570           31.061167  19.012600   
3                 0.113933  0.020142  0.017943           31.082365  19.015587   
4                 0.108888  0.018438  0.019454           31.019347  18.862127   

                     Right Ankle                                               \
            Acceleration (m/s^2)                     Angular Velocity (rad/s)   
          z                    x         y         z                        x   
0 -8.637949            -9.838516 -0.574557 -1.155564                -0.006665   
1 -8.560825            -9.851844 -0.576608 -1.153336                -0.003499   
2 -8.561372            -9.856227 -0.571967 -1.148940                -0.003592   
3 -8.637877            -9.849360 -0.574299 -1.151208                -0.002224   
4 -8.912165           

In [7]:
# remove units and simplify column titles
columns_str = ["_".join(df_tmp.columns[i]).replace(" ", "") for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = columns_str

l_replace = [df_tmp.columns[i].replace('(m/s^2)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(rad/s)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(uT)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace
df_tmp.head()

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,LeftFoot_Acceleration_x,LeftFoot_Acceleration_y,LeftFoot_Acceleration_z,LeftFoot_AngularVelocity_x,LeftFoot_AngularVelocity_y,LeftFoot_AngularVelocity_z,LeftFoot_MagneticField_x,LeftFoot_MagneticField_y,LeftFoot_MagneticField_z,RightFoot_Acceleration_x,RightFoot_Acceleration_y,RightFoot_Acceleration_z,RightFoot_AngularVelocity_x,RightFoot_AngularVelocity_y,RightFoot_AngularVelocity_z,RightFoot_MagneticField_x,RightFoot_MagneticField_y,RightFoot_MagneticField_z
0,0.007812,22.0,-9.846344,0.171822,1.227401,0.091794,-0.027342,-0.014275,46.815190,16.071594,13.902695,-9.764181,1.311812,0.985503,0.070709,0.084543,0.012162,55.879245,-1.209833,11.792871,-9.741885,-0.468814,-0.882530,0.118064,0.017438,0.021202,31.514212,18.934442,-8.637949,-9.838516,-0.574557,-1.155564,-0.006665,0.099050,0.009824,24.682745,-21.892916,18.683383,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.015624,22.0,-9.775612,0.050246,1.209962,0.094758,-0.032213,-0.006748,46.996207,16.216266,13.923646,-9.757018,1.316749,0.992521,0.067192,0.088808,0.013694,55.828066,-1.158919,11.847663,-9.729948,-0.520553,-0.887037,0.113823,0.015991,0.017926,31.112110,18.884349,-8.560825,-9.851844,-0.576608,-1.153336,-0.003499,0.100780,0.011414,24.586635,-21.917585,18.649075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.023437,22.0,-9.835819,0.125845,1.167702,0.097836,-0.038619,-0.010032,46.728334,16.212284,13.792188,-9.754793,1.332469,0.997156,0.065436,0.091731,0.010487,55.859044,-1.419488,11.842671,-9.728399,-0.466783,-0.891656,0.116528,0.015885,0.019570,31.061167,19.012600,-8.561372,-9.856227,-0.571967,-1.148940,-0.003592,0.095519,0.009830,24.610019,-21.911564,18.789673,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.031249,22.0,-9.810564,0.107856,1.203309,0.086923,-0.035422,-0.006915,46.646590,16.073072,13.775756,-9.757444,1.331871,0.987872,0.081388,0.086023,0.012167,55.757635,-1.280665,11.819464,-9.723669,-0.453419,-0.884954,0.113933,0.020142,0.017943,31.082365,19.015587,-8.637877,-9.849360,-0.574299,-1.151208,-0.002224,0.080491,0.011503,24.576289,-21.911981,18.789370,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.039061,22.0,-9.800482,0.050737,1.143245,0.091569,-0.041706,-0.007019,46.688921,16.195991,13.996201,-9.746306,1.337988,0.989957,0.081584,0.084634,0.013785,55.632360,-1.164697,11.939039,-9.716266,-0.484920,-0.887274,0.108888,0.018438,0.019454,31.019347,18.862127,-8.912165,-9.845130,-0.569882,-1.144508,-0.008403,0.088848,0.009861,24.728641,-21.912624,18.651281,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Obtain data with desired activity code

In [11]:
# subject running codes [11，12, 13, 14] = 1.2，1.8, 2,2, 2.7 m/s
AC_path = 'AC11'
df_1=df_tmp.loc[df_tmp['ActivityCode__']==11].dropna(axis=1,how='all')
df_2=df_tmp.loc[df_tmp['ActivityCode__']==12].dropna(axis=1,how='all')
df_3=df_tmp.loc[df_tmp['ActivityCode__']==12].dropna(axis=1,how='all')
df_1

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
46081,360.001562,11.0,-9.974068,-1.347072,0.677895,-0.220562,-0.228615,0.273134,44.722408,18.968927,17.878970,-9.991491,1.169622,0.316764,-0.391797,-0.026409,0.079752,54.486299,-0.700879,14.243443,-9.646737,-1.243761,-0.298544,-0.107500,0.115431,-1.096442,35.939250,8.139390,-7.839690,-10.184866,1.068869,-1.813298,-0.301764,-0.039599,0.602297,17.183016,-21.028860,22.267041
46082,360.009375,11.0,-10.014146,-1.199232,0.740735,-0.233258,-0.237026,0.277155,44.677540,18.875908,17.851687,-9.894322,1.129284,0.377750,-0.443868,-0.021077,0.074587,54.624316,-0.637001,14.153570,-9.339933,0.450489,-0.554461,-0.152345,0.121235,-1.122741,35.796610,8.518826,-8.102131,-10.045395,0.527543,-1.806480,-0.360286,-0.054417,0.607121,17.282826,-20.985699,22.290972
46083,360.017187,11.0,-10.009112,-1.214505,0.765358,-0.247386,-0.254473,0.285981,44.662798,18.935663,17.725205,-9.842143,1.034089,0.359816,-0.489293,-0.020051,0.056845,54.571590,-0.878128,14.195006,-9.205078,2.012951,-0.810118,-0.091060,0.142063,-1.099323,35.841830,8.735532,-7.963946,-10.142193,0.891481,-1.879800,-0.335305,-0.054269,0.629393,17.410718,-21.105488,22.316766
46084,360.024999,11.0,-10.066565,-1.058114,0.774009,-0.255267,-0.264608,0.296851,44.771132,18.697173,17.727407,-9.864354,1.050643,0.548582,-0.531757,-0.006066,0.068300,54.716611,-0.855829,14.072137,-9.376944,4.864260,0.311230,-0.034034,0.127130,-1.037657,35.757926,9.107138,-7.755243,-10.192590,0.824292,-1.688835,-0.347838,-0.060912,0.616715,17.257522,-21.143315,22.166832
46085,360.032811,11.0,-10.238797,-0.956435,0.751692,-0.303733,-0.276341,0.305328,44.858062,18.688865,17.854375,-9.730570,1.274111,0.703934,-0.611789,0.036148,0.048967,54.641266,-0.733628,13.980652,-10.330188,14.174714,1.152783,0.297181,0.087892,-0.638442,35.517882,9.462889,-7.639170,-10.282579,1.169958,-1.735477,-0.367345,-0.069602,0.651648,17.302458,-21.455428,22.216534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92158,719.964064,11.0,-9.794068,1.368866,-0.106957,0.234402,-0.337930,0.134014,47.543656,15.466692,15.172340,-10.770024,6.314946,0.277475,1.569673,0.319582,-0.225605,55.880841,-0.658505,12.337042,-6.140534,-0.406863,-4.718226,1.411633,0.581829,-1.986600,38.065177,12.425421,-2.101178,-18.780061,-4.183446,-8.147904,-2.168858,0.109240,3.900656,11.721643,-34.730797,17.674918
92159,719.971876,11.0,-8.956548,1.334049,-0.095037,0.203512,-0.218632,0.107485,47.514711,15.480236,14.969712,-10.026063,6.903624,0.619667,1.204710,0.398255,-0.178511,55.800364,-0.456728,12.537519,-6.038107,0.077595,-4.980594,1.426476,0.449075,-1.863531,37.878535,12.868328,-1.843153,-17.788694,-2.422326,-7.549400,-1.828666,0.127682,3.780727,10.978685,-35.584414,17.317269
92160,719.979688,11.0,-8.478771,1.300908,-0.192298,0.130408,-0.109959,0.080022,47.544787,15.468683,14.983420,-9.616588,6.949209,0.650762,0.727489,0.568922,-0.195238,55.416973,-0.356429,12.980098,-7.983144,-2.145962,-5.613421,1.181837,0.442711,-1.745738,37.719258,13.347016,-2.295385,-16.749235,-2.927795,-7.323452

### Simplify dataframe

In [12]:
# trim length of trial (number of rows in df)
obs_samples_trial = trial_length*60*frequency
df_1 = df_1.tail(obs_samples_trial)
df_2 = df_2.tail(obs_samples_trial) # keep last n samples to exclude speed transitions
df_3 = df_3.tail(obs_samples_trial)

# downsample trial
obs_samples_freq = int(128/frequency)

df_1 = df_1.iloc[::obs_samples_freq,:]
df_2 = df_2.iloc[::obs_samples_freq,:]
df_3 = df_3.iloc[::obs_samples_freq,:]
df_2

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
192953,1507.394242,12.0,0.669899,-2.223057,-1.920976,0.382160,0.018951,-0.190242,46.818495,16.133455,20.317440,-0.608555,2.186329,-2.453348,-1.200798,-0.190891,-0.859238,56.914883,-0.147788,13.506177,-16.837039,-1.113608,-3.173914,1.443275,-0.813301,-4.573894,16.201047,31.903772,-10.239838,-5.769559,16.215272,-29.334027,1.384635,1.170269,2.327332,29.566980,-10.954166,19.502649
192954,1507.402055,12.0,-0.387656,-2.260268,-1.068502,0.639635,-0.029822,-0.200912,46.640298,16.126301,20.766806,-2.552028,4.733920,-2.715482,-1.076356,-0.460825,-0.816853,56.892127,0.018744,13.332674,-18.180350,-1.333923,-5.275768,1.345168,-0.685515,-4.528758,15.318438,32.378329,-10.603190,-18.557201,4.409515,-34.629938,2.015977,1.830057,3.041721,29.150574,-11.261101,19.603632
192955,1507.409867,12.0,-2.585622,-2.271082,0.088246,0.856521,0.108481,-0.191238,46.356683,15.874559,20.944286,-6.018570,10.039680,-1.623083,-0.829050,-0.968216,-0.747826,57.124147,0.178464,12.907803,-19.704526,-0.715439,-7.125071,1.538587,-0.611216,-4.343658,14.333629,32.936853,-10.988315,-30.034481,-10.414625,-30.541549,2.010379,2.409682,3.639009,28.899816,-12.045292,19.701434
192956,1507.417679,12.0,-5.932279,-2.450753,0.260666,0.668717,0.346053,-0.094937,46.105430,15.410128,21.634286,-16.160978,19.740903,-0.448068,-0.537471,-1.498537,-0.734937,57.185280,0.359790,12.533050,-23.443022,1.174618,-5.585193,1.596861,-0.629626,-4.046760,13.182506,33.231829,-11.336208,-37.768251,1.329246,1.684808,1.457121,2.537937,4.997303,28.645632,-12.605279,20.523114
192957,1507.425491,12.0,-9.223000,-2.557179,-2.473370,0.804632,0.472711,0.098260,45.845252,15.745351,22.661836,-33.055878,24.773968,1.431991,-0.739346,-1.419947,-1.122380,57.261410,0.430477,12.143817,-24.260333,5.909558,-2.813163,0.799411,-0.765023,-3.706661,12.350512,33.585000,-11.843860,-41.018567,28.727346,21.823860,0.145407,2.984106,6.224279,28.220966,-13.209799,21.069269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231348,1807.343463,12.0,-9.353293,1.222394,1.625345,-0.142169,0.034614,-0.177125,50.843437,16.872015,10.648010,-9.088658,0.150357,1.608253,0.440451,0.272910,0.389406,57.418567,1.518668,14.197069,-10.376046,-0.368902,-2.140184,0.639215,-0.079757,-0.241094,32.060207,21.965298,-1.408134,-13.431479,1.032572,-0.510666,0.538763,-0.287644,-5.516171,23.632935,-31.388988,14.657065
231349,1807.351275,12.0,-9.335909,1.123459,1.540691,-0.148348,0.037640,-0.184895,50.714705,17.094845,10.726108,-8.898073,0.133779,1.682710,0.425002,0.231876,0.373033,57.305999,1.607555,14.145851,-10.403057,-0.227233,-2.169853,0.665515,-0.081127,-0.240671,31.953617,22.193238,-1.619954,-14.667708,1.500816,-1.404407,0.740622,-0.286241,-5.580673,24.313146,-30.173972,14.281123
231350,1807.359088,12.0,-9.171153,1.077339,1.206564,-0.128062,0.063893,-0.182946,50.640699,17.056767,10.871868,-8.852520,0.135237,1.756955,0.397494,0.157865,0.343987,57.243980,1.473774,14.260109,-10.562581,-0.652918,-2.124238,0.692417,-0.089017,-0.254645,31.504868,22.171354,-1.64088

Only include sensor data for model training and testing; remove time and activity code columns

In [13]:
df_1_data = df_1.iloc[:,2:] 
df_2_data = df_2.iloc[:,2:] 
df_3_data = df_3.iloc[:,2:] 
df_2_data

,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
192953,0.669899,-2.223057,-1.920976,0.382160,0.018951,-0.190242,46.818495,16.133455,20.317440,-0.608555,2.186329,-2.453348,-1.200798,-0.190891,-0.859238,56.914883,-0.147788,13.506177,-16.837039,-1.113608,-3.173914,1.443275,-0.813301,-4.573894,16.201047,31.903772,-10.239838,-5.769559,16.215272,-29.334027,1.384635,1.170269,2.327332,29.566980,-10.954166,19.502649
192954,-0.387656,-2.260268,-1.068502,0.639635,-0.029822,-0.200912,46.640298,16.126301,20.766806,-2.552028,4.733920,-2.715482,-1.076356,-0.460825,-0.816853,56.892127,0.018744,13.332674,-18.180350,-1.333923,-5.275768,1.345168,-0.685515,-4.528758,15.318438,32.378329,-10.603190,-18.557201,4.409515,-34.629938,2.015977,1.830057,3.041721,29.150574,-11.261101,19.603632
192955,-2.585622,-2.271082,0.088246,0.856521,0.108481,-0.191238,46.356683,15.874559,20.944286,-6.018570,10.039680,-1.623083,-0.829050,-0.968216,-0.747826,57.124147,0.178464,12.907803,-19.704526,-0.715439,-7.125071,1.538587,-0.611216,-4.343658,14.333629,32.936853,-10.988315,-30.034481,-10.414625,-30.541549,2.010379,2.409682,3.639009,28.899816,-12.045292,19.701434
192956,-5.932279,-2.450753,0.260666,0.668717,0.346053,-0.094937,46.105430,15.410128,21.634286,-16.160978,19.740903,-0.448068,-0.537471,-1.498537,-0.734937,57.185280,0.359790,12.533050,-23.443022,1.174618,-5.585193,1.596861,-0.629626,-4.046760,13.182506,33.231829,-11.336208,-37.768251,1.329246,1.684808,1.457121,2.537937,4.997303,28.645632,-12.605279,20.523114
192957,-9.223000,-2.557179,-2.473370,0.804632,0.472711,0.098260,45.845252,15.745351,22.661836,-33.055878,24.773968,1.431991,-0.739346,-1.419947,-1.122380,57.261410,0.430477,12.143817,-24.260333,5.909558,-2.813163,0.799411,-0.765023,-3.706661,12.350512,33.585000,-11.843860,-41.018567,28.727346,21.823860,0.145407,2.984106,6.224279,28.220966,-13.209799,21.069269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231348,-9.353293,1.222394,1.625345,-0.142169,0.034614,-0.177125,50.843437,16.872015,10.648010,-9.088658,0.150357,1.608253,0.440451,0.272910,0.389406,57.418567,1.518668,14.197069,-10.376046,-0.368902,-2.140184,0.639215,-0.079757,-0.241094,32.060207,21.965298,-1.408134,-13.431479,1.032572,-0.510666,0.538763,-0.287644,-5.516171,23.632935,-31.388988,14.657065
231349,-9.335909,1.123459,1.540691,-0.148348,0.037640,-0.184895,50.714705,17.094845,10.726108,-8.898073,0.133779,1.682710,0.425002,0.231876,0.373033,57.305999,1.607555,14.145851,-10.403057,-0.227233,-2.169853,0.665515,-0.081127,-0.240671,31.953617,22.193238,-1.619954,-14.667708,1.500816,-1.404407,0.740622,-0.286241,-5.580673,24.313146,-30.173972,14.281123
231350,-9.171153,1.077339,1.206564,-0.128062,0.063893,-0.182946,50.640699,17.056767,10.871868,-8.852520,0.135237,1.756955,0.397494,0.157865,0.343987,57.243980,1.473774,14.260109,-10.562581,-0.652918,-2.124238,0.692417,-0.089017,-0.254645,31.504868,22.171354,-1.640882,-15.673375,2.739823,-2.189711,1.022669,-0.289755,-5.606573,25.459896,-28.923799,14.199359
231351,-9.017789,1.329124,1.277886,-0.073453,0.047206,-0.184080,50.525998,17.

## Main For Dataset

In [15]:
# convert pandas dataframe to numpy array
load_X = df_1_data.to_numpy()
#load_X = np.concatenate((df_2_data, df_1_data), axis=0)
load_XT = df_2_data.to_numpy()
load_X.shape, load_XT.shape

((38400, 36), (38400, 36))

## Set up sensors

In [16]:
from random import choice

lags = frequency # length of trajectory used to train LSTM; chose 128 for Ingraham data sampled at 128 Hz
n = load_X.shape[0] # total number of time steps (observations)
m = load_X.shape[1] # number of features per time step

time = np.arange(1, n+1, 1)

## Visualize IMU data

Observing raw data is important for understanding what is being used to train and test models. We visualize data using altair (alt). Two tutorials on some basic functionality are linked below:

* Long tutorial (1hr): https://youtu.be/umTwkgQoo_E

* Short tutorial (20min): https://youtu.be/o-nVM_FdIVc

Uncomment the line below when code is fully functioning to disable the 5000-row dataframe limit

In [17]:
# alt.data_transformers.disable_max_rows()

In [18]:
# set time to start at 0 (optional for clean viz)
time_zeroed = df_2.loc[:,"Time(s)__"] - df_2["Time(s)__"].iloc[0]

# view the first portion of the trial
df_2_data_reduced = df_2.head(1000)
df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)

# view the last portion of the trial
#df_2_data_reduced = df_2.tail(4000) 
#df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.tail(4000)

df_2_data_reduced.head()

/tmp/ipykernel_2720515/3481633552.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)


,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,Time_Zeroed(s)
192953,1507.394242,12.0,0.669899,-2.223057,-1.920976,0.382160,0.018951,-0.190242,46.818495,16.133455,20.317440,-0.608555,2.186329,-2.453348,-1.200798,-0.190891,-0.859238,56.914883,-0.147788,13.506177,-16.837039,-1.113608,-3.173914,1.443275,-0.813301,-4.573894,16.201047,31.903772,-10.239838,-5.769559,16.215272,-29.334027,1.384635,1.170269,2.327332,29.566980,-10.954166,19.502649,0.000000
192954,1507.402055,12.0,-0.387656,-2.260268,-1.068502,0.639635,-0.029822,-0.200912,46.640298,16.126301,20.766806,-2.552028,4.733920,-2.715482,-1.076356,-0.460825,-0.816853,56.892127,0.018744,13.332674,-18.180350,-1.333923,-5.275768,1.345168,-0.685515,-4.528758,15.318438,32.378329,-10.603190,-18.557201,4.409515,-34.629938,2.015977,1.830057,3.041721,29.150574,-11.261101,19.603632,0.007812
192955,1507.409867,12.0,-2.585622,-2.271082,0.088246,0.856521,0.108481,-0.191238,46.356683,15.874559,20.944286,-6.018570,10.039680,-1.623083,-0.829050,-0.968216,-0.747826,57.124147,0.178464,12.907803,-19.704526,-0.715439,-7.125071,1.538587,-0.611216,-4.343658,14.333629,32.936853,-10.988315,-30.034481,-10.414625,-30.541549,2.010379,2.409682,3.639009,28.899816,-12.045292,19.701434,0.015624
192956,1507.417679,12.0,-5.932279,-2.450753,0.260666,0.668717,0.346053,-0.094937,46.105430,15.410128,21.634286,-16.160978,19.740903,-0.448068,-0.537471,-1.498537,-0.734937,57.185280,0.359790,12.533050,-23.443022,1.174618,-5.585193,1.596861,-0.629626,-4.046760,13.182506,33.231829,-11.336208,-37.768251,1.329246,1.684808,1.457121,2.537937,4.997303,28.645632,-12.605279,20.523114,0.023437
192957,1507.425491,12.0,-9.223000,-2.557179,-2.473370,0.804632,0.472711,0.098260,45.845252,15.745351,22.661836,-33.055878,24.773968,1.431991,-0.739346,-1.419947,-1.122380,57.261410,0.430477,12.143817,-24.260333,5.909558,-2.813163,0.799411,-0.765023,-3.706661,12.350512,33.585000,-11.843860,-41.018567,28.727346,21.823860,0.145407,2.984106,6.224279,28.220966,-13.209799,21.069269,0.031249


Select which sensor location to visualize.

In [19]:
location = 'RightAnkle' # RightAnkle, LeftAnkle, Chest, Waist

In [20]:
# Define signal types and axes.
sensor = ['Acceleration', 'AngularVelocity', 'MagneticField']
dir = ['x','y','z']
plotStack = [0,0,0] # Preallocate plot for each signal

# Generate plots for each sensor type
for iSensor in range(len(sensor)): # loop through the signal types
    # create plots for x,y,z directions
    x_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_x', title = sensor[iSensor]),
        color = alt.value('#c6dbef')
    ).properties(
        width = 1000,
        height = 200
    )
    y_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_y', title = sensor[iSensor]),
        color = alt.value("#6baed6")
    )
    z_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_z', title = sensor[iSensor]),
        color = alt.value("#08519c")
    ).interactive()
    # Combine x,y,z plots
    plotStack[iSensor] = x_signal + y_signal + z_signal

alt.vconcat(plotStack[0], plotStack[1], plotStack[2]).properties(title = [location,""])

alt.VConcatChart(...)

# SHRED model function

In [21]:
### Generate input sequences to a SHRED model
def train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags):
  """
    Trains SHRED model for time series reconstruction

  Args: 
    transformed_X (numpy array): MinMax scaled dataset.
    sc (MinMaxScaler): Fitted MinMax scaler for inverse transformation.
    train_indices (array): Indices for the training set.
    valid_indices (array): Indices for the validation set.
    test_indices (array): Indices for the test set.
    sensor_locations (array): Column indices for sensor data.
    num_sensors (int): Number of signal measurements from sensors (e.g, triaxial = 3)
    m (int): Number of features per timestep
    n (int): Total number of time steps (observations)
    lags (int): length of trajectory
    
  Return:
    test_recons: Reconstructed data from the SHRED model on the test set
    test_ground_truth: Ground truth data from the test set
  """

  all_data_in = np.zeros((n - lags, lags, num_sensors))
  for i in range(len(all_data_in)):
      all_data_in[i] = transformed_X[i:i+lags, sensor_locations]
  ### Generate training validation and test datasets both for reconstruction of states and forecasting sensors
  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
  valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
  test_data_in = torch.tensor(all_data_in[test_indices], dtype=torch.float32).to(device)

  ### -1 to have output be at the same time as final sensor measurements
  train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
  valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
  test_data_out = torch.tensor(transformed_X[test_indices + lags - 1], dtype=torch.float32).to(device)

  train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
  valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
  test_dataset = TimeSeriesDataset(test_data_in, test_data_out)


  ##Modify this part for TCN
  #shred = models_TCN.SHRED_TCN(num_sensors, m, number_channels=2, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
  tcn_chaneels = [64, 64, 64]
  shred = models_TCN.SHRED_TCN(num_sensors, m, num_channels=tcn_chaneels, fc_layers=[350, 400], kernel_size=7, dropout=0.1).to(device)

  validation_errors = models_TCN.fit(shred, train_dataset, valid_dataset, batch_size=64, num_epochs=500, lr=1e-3, verbose=True, patience=3)

  # Generate reconstructions from the test set and print mean square error compared to the ground truth
  test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
  test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())

  return test_recons, test_ground_truth

# Train models

Divide the data into training, validation and test.

In [22]:
# partition into training, validation, test sets
#train_indices, valid_indices, test_indices = ft.partition_data_seq(load_X, n, lags) 
train_indices, valid_indices, _ = ft.partition_data_seq(load_X, n, lags) 
test_data = load_XT


# normalize input data using MinMaxScaler
transformed_X, sc = ft.transform_data(load_X, train_indices) 
transformed_XT = sc.transform(test_data)
test_indices = np.arange(transformed_XT.shape[0])

### Define input sensor

In [23]:
# choose input sensor location
sensor_place = 'RightAnkle' # RightAnkle, Waist, or Chest

# choose input sensor type
sensor_path = '3acc_Training' # 3acc_Training, 3gyro_Training, 3acc3gyro_Training, or Xacc_Training

# access columns indices from main dataframe
sensor_locations, num_sensors = ft.sensor_loc_fun(sensor_path, sensor_place, df_2_data) # This function is specific to the dataset used in this project. Update it according the the types of signals (joint angles, EMG, etc) in your dataset.
train_names = [df_2_data.columns[i] for i in sensor_locations]

print('Number of signals: ', num_sensors)
print('Signals were chosen at: ', sensor_place)
print('Signals chosen: ', [df_2_data.columns[i] for i in sensor_locations])

Number of signals:  3
Signals were chosen at:  RightAnkle
Signals chosen:  ['RightAnkle_Acceleration_x', 'RightAnkle_Acceleration_y', 'RightAnkle_Acceleration_z']


In [24]:
# check path for saving dataframes
if trial_length == 6 and frequency == 128: # full-length trial, full frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ytest_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
else: # reduced trial length or frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'

print(os.path.isdir(save_test_df))
print(save_train_df)
print(save_test_df)

False
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC11/RightAnkle/3acc_Training/P01_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC11/RightAnkle/3acc_Training/P01_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv


In [25]:
test_data_length = len(load_XT)
test_time_indices = np.arange(0, test_data_length)
test_times = df_3_data.iloc[test_time_indices,0].to_numpy()

In [26]:
test_times.shape, test_time_indices.shape, load_XT.shape

((38400,), (38400,), (38400, 36))

In [27]:
# train SHRED model
Ypred, Ytest = train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags)

df_Ytest_SHRED = pd.DataFrame(Ytest, columns = df_2_data.columns)
df_Ypred_SHRED = pd.DataFrame(Ypred, columns = df_2_data.columns)

df_Ytest_SHRED['Type']='Measured'
df_Ypred_SHRED['Type']='TCN'

df_Ytest_SHRED['Time']=  test_times          # df_2.iloc[test_indices + lags - 1,0].to_numpy()
df_Ypred_SHRED['Time']=  test_times         # df_2.iloc[test_indices + lags - 1,0].to_numpy()

# save dataframes as .csv if specified
if save_df == True:
  df_Ytest_SHRED.to_csv(save_train_df)
  df_Ypred_SHRED.to_csv(save_test_df)


IndexError: index 38272 is out of bounds for axis 0 with size 38272

# Visualize Results

In [24]:
df_SHRED_tidy = ft.concatRaw(1,sensor_place, sensor_path, 1, 1, df_Ypred_SHRED, df_Ytest_SHRED ,subj)

In [25]:
print("df_SHRED_tidy 的列名:", df_SHRED_tidy.columns)
print("df_SHRED_tidy 的头部数据:\n", df_SHRED_tidy.head())

df_SHRED_tidy 的列名: Index(['Subject', 'Pred', 'True', 'Input Location', 'Sensor Type',
       'Output Location', 'Output Signal', 'Output Direction', 'Output Axis',
       'Assessment', 'Time'],
      dtype='object')
df_SHRED_tidy 的头部数据:
   Subject       Pred      True Input Location             Sensor Type  \
0      01 -20.656181 -18.96817     RightAnkle  Triaxial Accelerometer   
1      01  -1.308354  2.577638     RightAnkle  Triaxial Accelerometer   
2      01  -1.233435  2.359172     RightAnkle  Triaxial Accelerometer   
3      01    0.55952  2.237653     RightAnkle  Triaxial Accelerometer   
4      01  -0.366018 -0.298168     RightAnkle  Triaxial Accelerometer   

  Output Location     Output Signal Output Direction Output Axis  \
0           Waist      Acceleration         Vertical           x   
1           Waist      Acceleration               AP           y   
2           Waist      Acceleration               ML           z   
3           Waist  Angular Velocity         Vertica

In [26]:
# format: ft.extractSignal(output_location, output_signal, output_axis, df_SHRED_tidy)
    # output_location: 'Chest', 'Waist', 'RightAnkle', 'LeftAnkle'
    # output_signal: 'Acceleration', 'Angular Velocity', 'Magnetic Field'
    # output_axis: 'x', 'y', z'

Signal1 = ft.extractSignal('LeftAnkle', 'Angular Velocity', 'x', df_SHRED_tidy)
Signal2 = ft.extractSignal('Chest', 'Angular Velocity', 'x', df_SHRED_tidy)
Signal3 = ft.extractSignal('Waist', 'Angular Velocity', 'x', df_SHRED_tidy)

my_scheme = ['#1e88e5', "#6E6E6E"] # '#014337', '#1e88e5', '#DB1048'

# Compute error: ft.rmse_error, ft.mae_error, OR ft.mbe_error
Signal1_error = ft.rmse_error(Signal1[Signal1['Type'] == 'True']['Value'], Signal1[Signal1['Type'] == 'SHRED']['Value'])
Signal2_error = ft.rmse_error(Signal2[Signal2['Type'] == 'True']['Value'], Signal2[Signal2['Type'] == 'SHRED']['Value'])
Signal3_error = ft.rmse_error(Signal3[Signal3['Type'] == 'True']['Value'], Signal3[Signal3['Type'] == 'SHRED']['Value'])

# plot left ankle acceleration
line1 = alt.Chart(Signal1).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 1 RightAnkle: RMSE = {Signal1_error:.2f}'  # Can change this title to be specific to the output signal
)
# plot chest acceleration
line2 = alt.Chart(Signal2).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 2: Chest RMSE = {Signal2_error:.2f}'  # Can change this title to be specific to the output signal
)

# plot Waist acceleration
line3 = alt.Chart(Signal3).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 3: Waist RMSE = {Signal3_error:.2f}'  # Can change this title to be specific to the output signal
)

final_chart = alt.vconcat(line3, line2, line1).properties(
    title=f'Parameter = {save_tag}, Right Ankle Input' # Can change this title to match the input sensor
    # increase font size
).configure_axis(
    labelFontSize=18,
    titleFontSize=20
).configure_title(
    fontSize=24
).configure_legend(
    labelFontSize=18,
    titleFontSize=20
)

final_chart

alt.VConcatChart(...)